# Financial RAG：Colab GPU
先把程式上傳 GitHub，再填入儲存庫網址。這份 Notebook 不包含金鑰、財報或既有索引。
在 Colab「執行階段 → 變更執行階段類型」選 GPU；先從批次 1 開始。
以下每格由你按執行，不會自動啟動實驗。Colab 中斷前請完成最後的備份格。


In [ ]:
from pathlib import Path
import os, sys, subprocess, re
REPOSITORY = "" # @param {type:"string"}
BRANCH = "main" # @param {type:"string"}
DEVICE = "auto" # @param ["auto", "cpu", "cuda"]
ROOT = Path('/content/Financial_RAG')
if not re.fullmatch(r'https://github\.com/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+(?:\.git)?', REPOSITORY):
    raise ValueError('請填入 https://github.com/擁有者/儲存庫；勿把 token 放網址')
if not ROOT.exists():
    subprocess.run(['git','clone','--branch',BRANCH,'--',REPOSITORY,str(ROOT)],check=True)
elif not (ROOT/'colab/runtime_setup.py').is_file():
    raise ValueError('工作目錄已存在但不是此專案，請使用新的執行階段')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT/'Code'))
from colab.runtime_setup import install_dependencies
install_dependencies(ROOT)
print('套件安裝完成。若 Colab 提示重啟，請重啟後重跑本格；不會覆寫現有程式。')


In [ ]:
from colab.runtime_setup import configure
print(configure(DEVICE))
# 金鑰放左側「密鑰」，並允許此 Notebook 讀取；無金鑰仍可執行本機 BGE。
from google.colab import userdata
for name in ('VOYAGE_API_KEY','OPENAI_API_KEY','GEMINI_API_KEY'):
    try:
        value = userdata.get(name)
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        continue
    if value:
        os.environ[name] = value
print('金鑰載入完成（不顯示內容）')


## 資料
把財報或階段產物放到 Drive 的 `Financial_RAG_data`，保留原專案內的資料夾結構。
這格只複製資料到 `/content`，運作中的 Chroma／題庫不直接放 Drive。
既有索引跨 Windows/Linux 搬移仍需實際確認相容性；可改用 chunk JSON 在 Colab 新建索引。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_FOLDER = Path('/content/drive/MyDrive/Financial_RAG_data')
from colab.runtime_setup import copy_inputs
print('新增檔案數：', copy_inputs(ROOT, DATA_FOLDER))


## 執行階段範圍
先列出來源，再把要執行的 `id` 貼到下一格 `SOURCE_IDS`。起訖階段可相同，例如只做 5。
第五階段預設 BGE；Voyage 為遠端 API，不使用 Colab GPU，也仍可能產生費用。


In [ ]:
START_STAGE = 4 # @param {type:"integer"}
END_STAGE = 5 # @param {type:"integer"}
from config.settings import Settings
from functions.web_ui.catalog import catalog
STAGES = ('raw','processed','tagged','chunk','embed')
if not 1 <= START_STAGE <= END_STAGE <= 5:
    raise ValueError('須符合 1 ≤ 起始階段 ≤ 結束階段 ≤ 5')
settings = Settings(project_root=ROOT)
sources = catalog(settings, STAGES[START_STAGE-1])
for item in sources:
    print(item['id'])
print('來源數：',len(sources))


In [ ]:
FINAL_OUTPUT_ONLY = False # @param {type:"boolean"}
SOURCE_IDS = [] # 複製上面 id，例如 ['Financial_.../2024_2330_tagged.md']
CHUNK_MODE = 'parent_child' # @param ["parent_child", "original"]
CHUNK_SIZE = 250 # @param {type:"integer"}
TEXT_LABELS = 'body' # @param ["body", "description", "full"]
TABLE_LABELS = 'full' # @param ["body", "description", "full"]
BACKEND = 'bge' # @param ["bge", "voyage"]
ALLOW_VOYAGE_API = False # @param {type:"boolean"}
INDEX_NAME = 'Colab_BGE_250' # @param {type:"string"}
import time
from functions.web_ui.server import App
request = dict(stage=STAGES[START_STAGE-1], end_stage=STAGES[END_STAGE-1],
    ids=SOURCE_IDS, final_output_only=FINAL_OUTPUT_ONLY, ocr=False, chunk_mode=CHUNK_MODE, recursive_threshold=CHUNK_SIZE,
    text_embedding_mode=TEXT_LABELS, table_embedding_mode=TABLE_LABELS,
    backend=BACKEND, allow_voyage_api=ALLOW_VOYAGE_API, check_only=False,
    index_mode='new', index_name=INDEX_NAME)
app = App(settings)
app.start(request)
try:
    last = None
    while app.busy:
        with app.lock:
            state = {key: app.state.get(key) for key in ('state','stage','current','completed','total')}
        if state != last:
            print(state, flush=True)
            last = state
        time.sleep(2)
except KeyboardInterrupt:
    app.cancel()
    print('已請求停止；建庫會等目前文件完成後停止，請等 app.busy=False 再備份。')
    raise
print(app.state)


## 檢索與本機重排序
先執行索引清單，再把 collection 名稱填入下一格。重排序模型為 `BAAI/bge-reranker-v2-m3`。
查詢與階段工作不要同時執行，以免模型共同占滿 GPU 記憶體。


In [ ]:
subprocess.run([sys.executable, '-m', 'Code_2.main', '--list-indexes'], check=True)


In [ ]:
COLLECTION = '' # @param {type:"string"}
QUESTION = '' # @param {type:"string"}
if not COLLECTION or not QUESTION.strip():
    raise ValueError('請填入 collection 與問題')
subprocess.run([sys.executable,'-m','Code_2.main','--ask',QUESTION,
    '--collection',COLLECTION,'--backend',BACKEND,'--qa-mode','rerank',
    '--rerank-profile','local_bge_reranker_v2_m3'] + (['--allow-voyage-api'] if BACKEND=='voyage' else []),check=True)


In [ ]:
if 'app' in globals() and app.busy:
    raise RuntimeError('工作尚未結束，請等程序關閉資料庫後再備份')
from colab.runtime_setup import backup
print('備份：', backup(ROOT, DATA_FOLDER/'backups'))
